# 🦁 Lion-1 — تدريب LoRA على Google Colab

هذا الدفتر يشغّل مسار Lion-1 كاملاً على **T4 مجانية**:
1. فحص البيئة والبيانات (بلا شبكة).
2. تدريب LoRA فوق `Qwen/Qwen2.5-1.5B-Instruct` بصيغة أدوات LionAI الحقيقية.
3. دمج المحوّل وتصديره، ثم (اختياري) تحويله إلى GGUF Q4_K_M.
4. نشر النموذج على Hugging Face Hub وإنتاج رابط جاهز للتطبيق.
5. اختبار حي لأربعة مجسات (هوية/بلا تسريب/أداة صالحة/سؤال معرفي).

**قبل التشغيل:** `Runtime → Change runtime type → T4 GPU`.
**الحالة الصادقة:** البيانات الحالية عيّنة بذرية (~75 مثالاً) تعلّم الهوية وصيغة الأدوات، ولا تُكسب معرفة جديدة. وسّعها أولاً إن أردت جودة أعلى.

In [ ]:
!nvidia-smi || echo 'لا يوجد GPU — غيّر نوع التشغيل إلى T4 قبل المتابعة'
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
REPO = 'https://github.com/starmaen/LionAI.git'
BRANCH = 'main'
!git clone --depth 1 -b $BRANCH $REPO /content/LionAI
%cd /content/LionAI/model
!pip -q install -r requirements.txt
!python -c "import trl, peft, transformers, bitsandbytes; print('trl', trl.__version__, '| peft', peft.__version__, '| transformers', transformers.__version__)"

In [ ]:
# فحص سلامة البيانات: ترتيب الأدوار + كتل [[TOOL]] + منع تسريب أسماء المزودات
!python eval_lion.py data data/train.jsonl data/val.jsonl

In [ ]:
# التدريب: 3 حقب، LoRA r=16، تكميم 4-bit. المدة المتوقعة على T4: 10–20 دقيقة لعيّنة صغيرة.
!python train_lora.py --base Qwen/Qwen2.5-1.5B-Instruct --epochs 3 --out out/lion-1-lora

In [ ]:
# دمج المحوّل في النموذج الأساسي (fp16) — الناتج صالح للتحويل إلى GGUF
!python merge_export.py --base Qwen/Qwen2.5-1.5B-Instruct --lora out/lion-1-lora --out out/lion-1-merged
!du -sh out/lion-1-lora out/lion-1-merged

In [ ]:
# (اختياري) تحويل إلى GGUF Q4_K_M — يستغرق ~10 دقائق ويحتاج بناء llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements.txt
!python /content/llama.cpp/convert_hf_to_gguf.py out/lion-1-merged --outfile out/lion-1.f16.gguf --outtype f16
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4
! /content/llama.cpp/build/bin/llama-quantize out/lion-1.f16.gguf out/lion-1.Q4_K_M.gguf Q4_K_M
!ls -lh out/*.gguf

In [ ]:
# نشر النموذج على Hugging Face Hub — يُنتج رابطاً جاهزاً للتطبيق
# 1) أنشئ توكن كتابة: https://huggingface.co/settings/tokens (نوع Write)
# 2) في Colab: أيقونة 🔑 يسار الشاشة ← New secret ← الاسم HF_TOKEN ← فعّل Notebook access
# 3) عدّل HF_REPO إلى <اسمك>/lion-1 ثم شغّل الخلية
import os, glob, subprocess, sys
HF_REPO = 'YOUR_NAME/lion-1'   # ← غيّر هذا
HF_TOKEN = os.environ.get('HF_TOKEN', '')   # من Colab Secrets
ggufs = sorted(glob.glob('out/*.gguf'), key=os.path.getsize)
if not ggufs:
    print('⚠️ لا يوجد ملف GGUF — نفّذ الخلية السابقة أولاً')
elif HF_REPO.startswith('YOUR_NAME'):
    print('⚠️ عدّل HF_REPO إلى <اسمك>/lion-1 أولاً')
elif not HF_TOKEN:
    print('⚠️ لا يوجد HF_TOKEN — أضفه من أيقونة 🔑 في الشريط الجانبي')
else:
    subprocess.run([sys.executable, 'publish_hf.py', '--gguf', ggufs[-1],
                    '--repo', HF_REPO, '--token', HF_TOKEN], check=False)


In [ ]:
# اختبار حي: نشغّل llama-server في الخلفية ثم نمرّر المجسات الأربعة
import subprocess, time, urllib.request, os
srv = subprocess.Popen(['/content/llama.cpp/build/bin/llama-server', '-m', 'out/lion-1.Q4_K_M.gguf',
                        '--alias', 'lion-1', '--host', '127.0.0.1', '--port', '8089', '-c', '8192', '--jinja'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8089/health', timeout=2); break
    except Exception:
        time.sleep(2)
else:
    print('⚠️ لم يبدأ الخادم؛ تخطَّ الاختبار الحي')
!python eval_lion.py live --url http://127.0.0.1:8089/v1 --model lion-1
srv.terminate()

In [ ]:
# حفظ النواتج على Google Drive (اختياري)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lion-1 && cp -r out/lion-1-lora out/lion-1-merged /content/drive/MyDrive/lion-1/ 2>/dev/null; cp out/*.gguf /content/drive/MyDrive/lion-1/ 2>/dev/null; ls -lh /content/drive/MyDrive/lion-1

## بعد التدريب
- نزّل `lion-1.Q4_K_M.gguf` وشغّله محلياً: `./serve.sh lion-1.Q4_K_M.gguf` أو `ollama create lion-1 -f Modelfile`.
- في تطبيق LionAI أضف مزوداً مخصصاً (OpenAI-Compatible): `http://127.0.0.1:8089/v1` والموديل `lion-1`.
- أو الصق رابط Hugging Face الذي طبعه `publish_hf.py` في بطاقة 🦁 محرّك Lion-1.
- إن ظهر خطأ في أسماء معاملات `SFTConfig` فسببه اختلاف إصدار TRL؛ ثبّت `trl==0.12.*` أو عدّل الاسم حسب التوثيق.